# EDA Dataset Titanic — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (lebih mendalam dari `minggu_05.ipynb`): eksplorasi tabel, missing values, distribusi, hubungan dengan target `survived`, dan ringkasan hipotesis untuk pemodelan.

**Konteks data:** RMS Titanic (1912); dataset berisi **891** catatan penumpang. Target biner: `survived` (0 = meninggal, 1 = selamat).

**Sumber data:** `sns.load_dataset("titanic")` (Seaborn, selaras praktikum Minggu 2–5). Versi [Kaggle Titanic](https://www.kaggle.com/competitions/titanic/data) memiliki kolom serupa (`PassengerId`, `Cabin`, …); pola EDA sama.

**Pertanyaan analitik:**
- Bagaimana distribusi umur dan tarif (`fare`)?
- Apakah kelas kabin (`class` / `pclass`) berkaitan dengan survival?
- Apakah pola "women and children first" terlihat pada `sex` dan `who`?
- Kolom mana yang banyak missing, dan apa implikasinya untuk cleaning?

**Referensi:**
- [Towards Data Science — EDA with Python 101](https://towardsdatascience.com/exploratory-data-analysis-with-python-101-6349c2635b6a/)
- [Towards Data Science — Exploring Patterns of Survival](https://towardsdatascience.com/exploring-patterns-of-survival-from-the-titanic-dataset/)
- [DataCamp — Kaggle EDA & ML](https://www.datacamp.com/tutorial/kaggle-machine-learning-eda)
- [Microsoft Learn — Explore data with Python](https://learn.microsoft.com/en-us/training/modules/explore-analyze-data-with-python/)
- [UW-Madison Nexus — Exploring the Titanic Dataset](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Notebooks/Titanic-Dataset.html)
- [Michael Allen — Kaggle Titanic preprocessing](https://michaelallen1966.github.io/titanic/01_preprocessing.html)
- [Kaustubh Saha — Titanic survivor analysis](https://kaustubhsaha.postach.io/post/data-analysis-and-visualization-using-python-titanic-survivor-dataset)
- Modul PDF: `modul-05.tex` (Minggu 5 praktikum)

## 0. Kerangka CRISP-DM dan kamus fitur

Pada fase **Data Understanding** (CRISP-DM), kita memetakan arti setiap kolom sebelum memplot. EDA bersifat **iteratif**: temuan di sini dapat mengarahkan ulang cleaning (`minggu_03`) dan pemodelan (`minggu_06`).

| Kolom | Arti singkat |
|-------|----------------|
| `survived` | Target: 0 = meninggal, 1 = selamat |
| `pclass` | Kelas tiket 1/2/3 (1 = First) |
| `sex` | Jenis kelamin |
| `age` | Umur (tahun); banyak NA |
| `sibsp` | Jumlah saudara/pasangan di kapal |
| `parch` | Jumlah orang tua/anak di kapal |
| `fare` | Tarif yang dibayar |
| `embarked` | Pelabuhan naik: C/Q/S |
| `class` | Label kelas (First/Second/Third); setara `pclass` |
| `who` | man / woman / child |
| `adult_male` | True jika penumpang laki-laki dewasa |
| `deck` | Dek kabin; sangat banyak NA |
| `embark_town` | Nama kota pelabuhan; setara `embarked` |
| `alive` | yes/no; turunan `survived` |
| `alone` | True jika bepergian tanpa keluarga (sibsp+parch=0) |

**Kolom yang dipakai dominan di notebook ini:** `survived`, `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`, `class`, `who`, `alone`, `deck` (eksploratif). Kolom redundan diverifikasi di bagian audit.

## 1. Persiapan lingkungan

Import pustaka standar praktikum dan atur tema Seaborn agar plot konsisten di sel berikutnya.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

**Memuat data.** `.copy()` membuat salinan independen agar transformasi di notebook ini tidak mengubah cache dataset global Seaborn.

In [ ]:
df = sns.load_dataset("titanic").copy()
print("Bentuk data:", df.shape)
df.head()

### 1b. Audit kualitas data

`sample` memberi cuplikan acak; cek duplikat baris; verifikasi kolom yang isinya hampir identik agar tidak menganalisis informasi ganda.

In [ ]:
RANDOM_STATE = 42  # reproduktifitas, selaras notebook minggu lain

print("Cuplikan acak 20 penumpang:")
display(df.sample(20, random_state=RANDOM_STATE))

n_dup = df.duplicated().sum()
print(f"\nBaris duplikat penuh: {n_dup}")

# survived (0/1) vs alive (no/yes)
alive_as_int = df["alive"].map({"no": 0, "yes": 1})
print("survived vs alive identik?", df["survived"].equals(alive_as_int))

# embarked (kode) vs embark_town (nama)
emb_map = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
emb_mapped = df["embarked"].map(emb_map)
match_town = emb_mapped.equals(df["embark_town"]) | (
    emb_mapped.fillna("__NA__") == df["embark_town"].fillna("__NA__")
).all()
print("embarked (dipetakan) selaras embark_town?", match_town)

# pclass vs class
class_to_pclass = {"First": 1, "Second": 2, "Third": 3}
print("pclass selaras class?", df["pclass"].equals(df["class"].map(class_to_pclass)))


**Interpretasi (audit):** Tidak ada duplikat baris penuh pada versi Seaborn ini; `alive` dan `embark_town` redundan dengan `survived` dan `embarked`. Untuk EDA berikutnya kita tetap memakai `survived`, `embarked`, `class`/`pclass` tanpa menghapus kolom (agar Anda bisa membandingkan sendiri).

**Cuplikan akhir dan ringkasan kolom.** `tail()` memeriksa baris terakhir; `info()` menampilkan tipe dan jumlah non-NA per kolom (setara `str()` di R).

In [ ]:
display(df.tail())
df.info()

**Statistik deskriptif.** `describe()` untuk numerik; `describe(include="object")` untuk kategorikal (frekuensi modus, unique, top).

In [ ]:
display(df.describe())
display(df.describe(include="object"))

### 2a. Membaca `describe`: mean vs median dan kategori langka

- **Mean > median** pada `fare` → skew kanan (outlier tarif tinggi).
- **Mean ≈ median** pada `age` (setelah drop NA) → distribusi relatif simetris untuk sebagian besar penumpang.
- **Kategori langka:** level `embarked` Q dan C jauh lebih sedikit daripada S; ini relevan untuk encoding nanti.

In [ ]:
num_summary = df[["age", "fare", "sibsp", "parch"]].agg(["mean", "median", "std", "min", "max"])
display(num_summary.round(2))

print("\nKategori dengan frekuensi < 5% dari n (indikasi langka):")
for col in ["embarked", "who", "deck"]:
    vc = df[col].value_counts(normalize=True, dropna=False)
    rare = vc[vc < 0.05]
    if len(rare):
        print(f"\n{col}:")
        print((rare * 100).round(1).astype(str) + "%")


## 2. EDA tabular (Part I)

Eksplorasi berbasis tabel sebelum visual: tipe data, kardinalitas, missing, frekuensi kategori, dan survival rate per grup.

In [ ]:
print("Tipe data per kolom:")
print(df.dtypes)
print("\nJumlah nilai unik per kolom:")
df.nunique().sort_values(ascending=False)

**Missing values.** Menghitung jumlah dan persentase NA; kolom `deck` dan `age` biasanya paling bermasalah pada versi Seaborn.

In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df) * 100).round(2)
missing_tbl = pd.DataFrame({"jumlah_na": missing_count, "persen": missing_pct})
missing_tbl[missing_tbl["jumlah_na"] > 0]

**Visual missing.** Barplot memudahkan membandingkan skala missing antar kolom.

In [ ]:
miss = missing_tbl[missing_tbl["jumlah_na"] > 0].sort_values("jumlah_na", ascending=True)
plt.figure(figsize=(8, 4))
sns.barplot(x=miss["jumlah_na"], y=miss.index, hue=miss.index, palette="Blues_d", legend=False)
plt.title("Jumlah nilai hilang (NA) per kolom")
plt.xlabel("Jumlah NA")
plt.ylabel("Kolom")
plt.tight_layout()
plt.show()

**Interpretasi (missing):** Kolom dengan NA tinggi (`deck`) sulit dipakai tanpa imputasi atau penghapusan; `age` perlu strategi imputasi sebelum pemodelan (lihat `minggu_03.ipynb`).

**Heatmap pola missing.** Baris = penumpang (urutan indeks), kolom = fitur; hitam/terang menandakan NA.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Pola nilai hilang per kolom (setiap baris = satu penumpang)")
plt.xlabel("Kolom")
plt.ylabel("Indeks baris")
plt.tight_layout()
plt.show()

**Umur vs kategori `who` (petunjuk imputasi).** Distribusi `age` berbeda antara man/woman/child — strategi imputasi per grup lebih masuk akal daripada median global (implementasi di `minggu_03.ipynb`).

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="who", y="age", order=["child", "woman", "man"])
plt.title("Sebaran umur menurut who (man / woman / child)")
plt.xlabel("who")
plt.ylabel("age")
plt.tight_layout()
plt.show()

**Interpretasi (age vs who):** `child` konsentrasi umur rendah; `man` dan `woman` tumpang tindih tetapi median berbeda. Missing `age` dapat diisi dengan median per `who`, bukan satu angka untuk seluruh kapal.

**Frekuensi kategorikal.** `value_counts()` menunjukkan distribusi level pada variabel kategorikal utama.

In [ ]:
for col in ["sex", "class", "embarked", "who"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

**Survival rate per grup.** `groupby(...).mean()` pada kolom biner `survived` setara proporsi yang selamat.

In [ ]:
survival_by = {}
for col in ["sex", "pclass", "class", "embarked", "who"]:
    survival_by[col] = df.groupby(col, observed=True)["survived"].mean().sort_values(ascending=False)
    print(f"\nTingkat survival menurut {col}:")
    print((survival_by[col] * 100).round(1).astype(str) + "%")

**Tabel silang.** `crosstab` dengan normalisasi baris menunjukkan proporsi survival dalam setiap kategori.

In [ ]:
ct_sex = pd.crosstab(df["sex"], df["survived"], normalize="index")
ct_sex.columns = ["meninggal_prop", "selamat_prop"]
print("Crosstab sex × survived (proporsi per baris):")
display(ct_sex.round(3))

ct_class = pd.crosstab(df["class"], df["survived"], normalize="index")
ct_class.columns = ["meninggal_prop", "selamat_prop"]
print("\nCrosstab class × survived:")
display(ct_class.round(3))

**Korelasi numerik.** Matriks Pearson pada fitur inti; urutkan menurut korelasi dengan `survived`.

In [ ]:
num_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[num_cols].corr()
display(corr.round(3))
print("\nKorelasi dengan survived (diurutkan):")
print(corr["survived"].drop("survived").sort_values(key=abs, ascending=False).round(3))

**Interpretasi (tabel, angka contoh):** Perempuan **~74%** selamat vs laki-laki **~19%**; kelas First **~63%**, Second **~47%**, Third **~24%**; `pclass` berkorelasi **-0,34** dengan `survived`. Pelabuhan Cherbourg (C) **~55%** vs Southampton (S) **~34%** survival.


### 2b. Ukuran keluarga dan bepergian sendiri

`family_size = sibsp + parch + 1` (diri sendiri + kerabat di kapal). Kolom `alone` sudah disediakan Seaborn; kita bandingkan survival.

In [ ]:
df_eda = df.copy()
df_eda["family_size"] = df_eda["sibsp"] + df_eda["parch"] + 1

print("Survival menurut alone:")
print((df_eda.groupby("alone")["survived"].mean() * 100).round(1).astype(str) + "%")

print("\nSurvival menurut family_size (nilai umum):")
fsurv = df_eda.groupby("family_size")["survived"].agg(["mean", "count"])
fsurv = fsurv[fsurv["count"] >= 10]
display((fsurv["mean"] * 100).round(1).to_frame("survival_pct"))

plt.figure(figsize=(6, 4))
sns.barplot(data=df_eda, x="alone", y="survived", estimator="mean", errorbar=None)
plt.title("Proporsi survival: traveling alone vs with family")
plt.xlabel("alone")
plt.ylabel("Proporsi selamat")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Interpretasi (keluarga):** Penumpang **tidak** sendirian (`alone=False`) cenderung survival sedikit lebih tinggi pada agregat ini; keluarga sangat besar (`family_size` ≥ 8) jarang dan tidak selalu selamat — nuansa berbeda dari stereotip "keluarga selalu dilindungi".

## 3. EDA univariat (Part II — distribusi)

Memahami bentuk distribusi setiap variabel sebelum menghubungkannya dengan target.

**Target `survived`.** Proporsi kelas mayoritas/minoritas.

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="survived")
plt.title("Distribusi target survived")
plt.xlabel("survived (0=meninggal, 1=selamat)")
plt.ylabel("Jumlah penumpang")
plt.tight_layout()
plt.show()
print("Proporsi selamat:", df["survived"].mean().round(3))

**Interpretasi:** Sekitar 38% penumpang selamat — kelas tidak seimbang (imbalance) yang relevan untuk metrik klasifikasi nanti.

**Distribusi numerik.** Histogram + KDE untuk `age` dan `fare`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["age"].dropna(), kde=True, ax=ax[0], color="steelblue")
ax[0].set_title("Distribusi umur (age)")
ax[0].set_xlabel("age")
ax[0].set_ylabel("Frekuensi")
sns.histplot(df["fare"], kde=True, ax=ax[1], color="coral")
ax[1].set_title("Distribusi tarif (fare)")
ax[1].set_xlabel("fare")
ax[1].set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()

**Interpretasi:** Umur cenderung unimodal dengan puncak dewasa muda; `fare` sangat skew (kanan) karena outlier kelas First — transformasi log dapat dipertimbangkan sebelum regresi/pemodelan.

**Kategorikal utama.** Countplot untuk `sex`, `class`, dan `embarked`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["sex", "class", "embarked"]):
    sns.countplot(data=df, x=col, ax=ax, order=df[col].value_counts().index)
    ax.set_title(f"Frekuensi {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Jumlah")
plt.tight_layout()
plt.show()

**Panel histogram numerik.** Ringkasan cepat semua kolom numerik (pola tutorial EDA 101).

In [ ]:
num_df = df.select_dtypes(include="number")
num_df.hist(bins=20, figsize=(10, 8), edgecolor="white")
plt.suptitle("Histogram semua variabel numerik", y=1.02)
plt.tight_layout()
plt.show()

### 4a. Interaksi jenis kelamin × kelas kabin

Pertanyaan: apakah perbedaan survival antara gender **sama** di setiap kelas? Pivot dan heatmap membaca proporsi selamat.

In [ ]:
pivot_sex_class = df.pivot_table(
    index="sex", columns="class", values="survived", aggfunc="mean", observed=True
)
print("Tingkat survival (proporsi) — sex × class:")
display((pivot_sex_class * 100).round(1))

plt.figure(figsize=(6, 4))
sns.heatmap(
    pivot_sex_class, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
    cbar_kws={"label": "Proporsi selamat"},
)
plt.title("Survival rate: sex × class")
plt.xlabel("class")
plt.ylabel("sex")
plt.tight_layout()
plt.show()

**Interpretasi (sex×class):** Perempuan di kelas First hampir selalu selamat pada sampel ini; laki-laki Third class survival terendah. Interaksi ini menunjukkan **satu fitur saja tidak cukup** — model perlu kombinasi `sex` dan `pclass`/`class`.

## 4. EDA bivariat — hubungan dengan survival

Setiap plot diikuti interpretasi singkat (persyaratan modul Minggu 5).

**Survival menurut jenis kelamin.** Barplot proporsi selamat per `sex`.

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="sex", y="survived", estimator="mean", errorbar=None)
plt.title("Proporsi survival menurut jenis kelamin")
plt.xlabel("sex")
plt.ylabel("Proporsi selamat")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Interpretasi:** Perempuan memiliki proporsi survival jauh lebih tinggi (~74% vs ~19% laki-laki pada agregat ini), konsisten dengan kebijakan evakuasi dan ukuran sampel.

**Survival menurut kelas kabin.** Countplot dengan `hue=survived`.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="class", hue="survived", order=["First", "Second", "Third"])
plt.title("Jumlah penumpang per kelas, diwarnai survival")
plt.xlabel("class")
plt.ylabel("Jumlah")
plt.legend(title="survived", labels=["meninggal (0)", "selamat (1)"])
plt.tight_layout()
plt.show()

**Interpretasi:** Kelas First memiliki proporsi selamat tertinggi; Third class dominan jumlah penumpang tetapi banyak yang tidak selamat.

**Tarif vs kelas.** Boxplot dan violinplot membandingkan sebaran `fare`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="class", y="fare", order=["First", "Second", "Third"], ax=ax[0])
ax[0].set_title("Boxplot fare per class")
ax[0].set_xlabel("class")
ax[0].set_ylabel("fare")
sns.violinplot(data=df, x="class", y="fare", order=["First", "Second", "Third"], ax=ax[1])
ax[1].set_title("Violinplot fare per class")
ax[1].set_xlabel("class")
ax[1].set_ylabel("fare")
plt.tight_layout()
plt.show()

**Interpretasi:** Median `fare` naik jelas dari Third ke First; outlier tinggi di First — relevan untuk feature scaling dan deteksi outlier (`minggu_03`).

### 4b. Outlier numerik dan transformasi log pada `fare`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["age", "sibsp", "parch"]):
    sns.boxplot(y=df[col].dropna() if col == "age" else df[col], ax=ax)
    ax.set_title(f"Boxplot {col}")
    ax.set_ylabel(col)
plt.suptitle("Deteksi outlier pada fitur numerik", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.kdeplot(df["fare"], fill=True, ax=ax[0])
ax[0].set_title("KDE fare (asli)")
ax[0].set_xlabel("fare")
df_eda["log_fare"] = np.log1p(df_eda["fare"])
sns.kdeplot(df_eda["log_fare"], fill=True, ax=ax[1], color="darkgreen")
ax[1].set_title("KDE log1p(fare)")
ax[1].set_xlabel("log(1 + fare)")
plt.tight_layout()
plt.show()

**Interpretasi (outlier & log):** `fare` punya ekor panjang ke kanan; `log1p` meratakan skala untuk eksplorasi/visual berikutnya. Outlier `age` (lanjut usia) dan `sibsp`/`parch` ekstrem perlu dipertimbangkan saat cleaning, bukan dihapus otomatis tanpa konteks.

**Umur vs tarif, diwarnai survival.** Scatter untuk melihat pemisahan kelas target di ruang 2D.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=0.65, palette="Set1")
plt.title("Umur vs tarif (hue: survived)")
plt.xlabel("age")
plt.ylabel("fare")
plt.tight_layout()
plt.show()

**Interpretasi:** Titik selamat (`survived=1`) lebih sering di rentang `fare` tinggi; pemisahan tidak sempurna — perlu fitur lain (mis. `sex`, `pclass`).

**Distribusi umur per gender.** Histogram bertumpuk (`multiple="stack"`).

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df.dropna(subset=["age"]), x="age", hue="sex", multiple="stack", kde=False)
plt.title("Distribusi umur per jenis kelamin")
plt.xlabel("age")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

**FacetGrid: umur vs outcome.** Satu histogram `age` per nilai `survived` (pola FacetGrid tutorial EDA).

In [ ]:
g = sns.FacetGrid(df.dropna(subset=["age"]), col="survived", col_wrap=2, height=3, sharex=True)
g.map_dataframe(sns.histplot, x="age", kde=True, bins=25)
g.set_axis_labels("age", "Frekuensi")
g.set_titles(col_template="survived={col_name}")
g.fig.suptitle("Distribusi umur menurut survival", y=1.05)
plt.show()

**Interpretasi:** Penumpang selamat sedikit lebih muda pada puncak distribusi; perbedaan umur saja tidak memisahkan kelas sekuat `sex` atau `class`.

**Survival per `who` dan pelabuhan.** `catplot` facet untuk eksplorasi kategorikal lanjutan.

In [ ]:
sns.catplot(data=df, x="who", y="survived", kind="bar", height=4, aspect=1.2, errorbar=None)
plt.suptitle("Proporsi survival menurut kategori who (man/woman/child)", y=1.02)
plt.show()

sns.catplot(data=df.dropna(subset=["embarked"]), x="embarked", y="survived", kind="bar", height=4, errorbar=None)
plt.suptitle("Proporsi survival menurut pelabuhan naik (embarked)", y=1.02)
plt.show()

**Interpretasi:** Kategori `woman` dan `child` survival rate tinggi; `embarked=C` (Cherbourg) sedikit lebih tinggi — bisa berkorelasi dengan proporsi penumpang First di pelabuhan itu.

### 5a. (Opsional) Korelasi setelah one-hot encoding

Cuplikan lanjutan menuju `minggu_04.ipynb`: kategorikal di-encode agar korelasi dengan `survived` terbaca di heatmap lebih luas. **Bukan** pipeline final.

In [ ]:
df_enc = pd.get_dummies(df_eda.drop(columns=["alive", "embark_town"], errors="ignore"), drop_first=True)
corr_enc = df_enc.corr(numeric_only=True)["survived"].drop("survived").sort_values(key=abs, ascending=False)
print("Top 12 fitur (encoded) terkait survived:")
display(corr_enc.head(12).round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(df_enc.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Heatmap korelasi — semua fitur setelah get_dummies (cuplikan)")
plt.tight_layout()
plt.show()

### 4c. `deck` (subset terisi) dan `adult_male`

In [ ]:
deck_surv = (
    df.dropna(subset=["deck"])
    .groupby("deck", observed=True)["survived"]
    .agg(["mean", "count"])
    .query("count >= 5")
    .sort_values("mean", ascending=False)
)
print("Survival per deck (min 5 penumpang, NA diabaikan):")
display((deck_surv["mean"] * 100).round(1).to_frame("survival_pct"))

plt.figure(figsize=(5, 4))
sns.barplot(data=df, x="adult_male", y="survived", estimator="mean", errorbar=None)
plt.title("Proporsi survival menurut adult_male")
plt.xlabel("adult_male")
plt.ylabel("Proporsi selamat")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print("Korelasi adult_male (as int) dengan survived:", df["adult_male"].astype(int).corr(df["survived"]).round(3))

**Interpretasi (deck & adult_male):** `deck` >75% NA — interpretasi hati-hati; pola deck C/E mungkin confounded dengan kelas. `adult_male` sangat berkorelasi dengan survival rendah dan hampir redundan dengan `sex`/`who` — saat modeling, hindari duplikasi fitur.

## 5. EDA multivariat — korelasi dan pasangan fitur

**Catatan:** Korelasi Pearson mengukur hubungan linear; bukan bukti kausalitas.

**Heatmap korelasi.** Hanya baris lengkap (tanpa NA) pada subset numerik.

In [ ]:
num = df[num_cols].dropna()
plt.figure(figsize=(7, 5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Heatmap korelasi Pearson (fitur numerik inti)")
plt.tight_layout()
plt.show()

**Interpretasi:** `pclass` dan `fare` berkorelasi dengan `survived`; `sibsp`/`parch` lemah — mengundang feature engineering `family_size`.

**Pairplot.** Scatter per pasangan fitur; `corner=True` mengurangi redundansi panel.

In [ ]:
cols = ["age", "fare", "pclass"]
sns.pairplot(
    df[cols + ["survived"]].dropna(),
    hue="survived",
    corner=True,
    plot_kws={"alpha": 0.6},
)
plt.show()

**Interpretasi:** Warna `survived=1` lebih terkonsentrasi pada `pclass` rendah (First) dan `fare` tinggi di beberapa panel — mendukung hipotesis klasifikasi berbasis kelas sosial dan tarif.

## 6. Eksplorasi fitur teks (opsional)

Dataset Seaborn di atas **tidak** menyertakan kolom `name`. Untuk demo ekstraksi gelar (pola umum di kompetisi Kaggle), kita muat versi CSV standar yang memiliki kolom `Name` dan `Survived`.

In [ ]:
# CSV publik (struktur mirip Kaggle); butuh koneksi internet sekali saat dijalankan
url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_kaggle = pd.read_csv(url_titanic)

df_kaggle["title"] = df_kaggle["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
title_survival = df_kaggle.groupby("title", observed=True)["Survived"].agg(["mean", "count"])
title_survival = title_survival[title_survival["count"] >= 5].sort_values("mean", ascending=False)
print("Survival rate per gelar dari Name (minimal 5 penumpang):")
display((title_survival["mean"] * 100).round(1).to_frame("survival_pct"))

**Interpretasi:** Gelar `Mrs`, `Miss`, `Master` cenderung survival tinggi; `Mr` rendah — informasi redundan dengan `sex`/`who` tetapi berguna untuk feature engineering lanjutan.

## 7. Ringkasan temuan, bias, dan hipotesis pemodelan

### Ringkasan temuan (EDA)

1. **Target tidak seimbang:** ~38% penumpang selamat (`survived=1`).
2. **Gender:** Perempuan ~74% vs laki-laki ~19% selamat; `who=woman/child` menguatkan pola evakuasi.
3. **Kelas sosial:** First ~63% > Second ~47% > Third ~24%; interaksi **sex×class** menunjukkan efek gender tidak homogen antar kelas.
4. **Tarif (`fare`):** Skew kanan; `log1p(fare)` membantu visualisasi; korelasi positif lemah dengan survival.
5. **Umur:** ~20% missing; imputasi per `who` lebih masuk akal daripada median global.
6. **Keluarga:** `alone=True` survival sedikit lebih rendah; `family_size` menengah kadang lebih baik.
7. **Missing & redundansi:** `deck` sparse; audit kolom mengurangi analisis ganda.
8. **Fitur turunan:** `family_size`, gelar dari `Name` (Kaggle CSV), encoding untuk modeling.

### Bias dan limitasi

- Sampel **891** penumpang tidak merepresentasikan seluruh kapal.
- Pola sosial 1912; model deskriptif, bukan kausal.
- Missing `age`/`deck` berpotensi bias jika imputasi sembarangan.
- EDA iteratif (CRISP-DM): ulangi setelah baseline model jika subgroup error tinggi.

### Hipotesis pemodelan

Klasifikasi `survived` dengan **`sex`**, **`pclass`**, **`fare`**, **`age`** (imputasi per `who`), interaksi **sex×class**. Hindari `adult_male` + `sex` + `who` bersamaan tanpa seleksi fitur.

### Langkah berikutnya

1. `minggu_03.ipynb` — cleaning  2. `minggu_04.ipynb` — encoding  3. `minggu_06.ipynb` — modeling

Jalankan **Kernel → Restart & Run All**.
